# A long run, end to end

Records `ai0` while `ao0` generates a square wave, both triggered by the same
PFI8 line that carries the rubidium 1 pps — so **scan 0 is an atomic second**.

**Wiring**

| signal | terminal | ground |
|---|---|---|
| 1 pps → PFI 8 | 81 | D GND 82 |
| AO 0 → AI 0 | 15 → 1 | AO GND 16 → AI GND 3 |

Restart the kernel before running this if you have edited the driver.

In [ ]:
import sys, time
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

DRIVER = Path(r"C:/path/to/src/drivers/DAQ/NIUSB-6289_v2")   # <-- edit me
sys.path.insert(0, str(DRIVER))

from usb6289 import USB6289
from postprocess import (load_long_run, load_segment, channels_of,
                         edge_report, clean_edges, times_from_edges)
from livescope import LiveScope

OUTDIR = Path(r"D:/data") / time.strftime("run_%Y%m%dT%H%M%SZ", time.gmtime())
print(OUTDIR)          # NOT inside OneDrive - a sync client will stall the write

## 1. Connect and configure

In [ ]:
daq = USB6289("daq", device="Dev1")

ai0, = daq.ai.setup({"ai0": (-2.0, 2.0)},          # +/-2 V on this pin
                    rate=25_000,
                    trigger="/Dev1/PFI8")

ao0, = daq.ao.setup({"ao0": {"wave": 1.0}},        # 1 V amplitude
                    shape="square", freq=100.0, rate=100_000,
                    trigger="/Dev1/PFI8")

daq.pps.terminal("/Dev1/PFI8")     # same line: scan 0 IS an atomic second
daq.pps.counter("ctr1")            # leaves ctr0 free

print(daq.ai.actual_rate(), "S/s actual")

## 2. Check before you start

`check()` finds what no single validator can see. Expect an informational note
that `ai.trigger` and `pps.terminal` are the same line — that is the intent.

`daq.ao.onboard()` **must be True**: it means the waveform loops from the
board's own memory, so a saturated AI stream cannot starve it over eight hours.

In [ ]:
daq.check(strict=True)             # raises if anything cannot run
print("AO regenerates onboard:", daq.ao.onboard(),
      f"(buffer {daq.ao.buffer_len():,} of {daq.ao.fifo_samples():,} samples)")
daq.wiring()

## 3. Rehearse — 3 s, look at it, before committing hours

In [ ]:
daq.ai.duration(3.0)
data, edges = daq.acquire_with_pps()
y = data["ai0"]

print(f"{len(y):,} samples, {y.min():+.3f} .. {y.max():+.3f} V")
edge_report(edges, nominal_rate=daq.ai.actual_rate())

t = daq.ai.time_axis()
plt.figure(figsize=(11, 3))
plt.plot(t[:2000], y[:2000], lw=0.8)
plt.xlabel("board time [s]"); plt.ylabel("V"); plt.title("first 80 ms")
plt.show()

## 4. Start the run

AO first, so it is armed before the AI task starts. `background=True` returns
immediately — this cell does not block, and the acquisition continues while you
run the cells below.

`rotate_minutes=None` gives **one file per channel** for the whole run. For runs
over a day use `rotate_minutes=60`, so a closed hourly file can be archived
while the run continues.

In [ ]:
daq.ao.start()                     # arms; fires on the next PFI8 edge

run = daq.long_run(OUTDIR,
                   hours=8,
                   rotate_minutes=None,
                   background=True)
run

## 5. Watch it — health check

`live_preview()` reads a rolling 60 s of the FIRST channel from RAM, decimated.
It answers "is it alive", nothing more. Re-run this cell whenever you like.

In [ ]:
s = run.status()
t, y = daq.live_preview()
print(f"{s['seconds']:,.0f} s  segment {s['segment']}  alive={run.alive}")

if len(y):
    plt.figure(figsize=(11, 3))
    plt.plot(t, y, lw=0.6)
    plt.xlabel("board time [s]"); plt.ylabel("V"); plt.show()

## 6. Watch it — the live plot

**Why the obvious loop shows nothing.** With the default `%matplotlib inline`
backend there is no GUI event loop, so `plt.pause()` redraws nothing and the
cell simply blocks until it ends. Two things have to happen every frame:

1. **clear the axes** — `ax.plot()` in a loop ADDS a line each time, so after a
   few hundred frames the figure is hundreds of lines and grinds to a halt;
2. **push the figure to the screen** — `display()` after `clear_output(wait=True)`
   on inline, or `plt.pause()` on an interactive backend.

`time.sleep()` alone never works on inline: it sleeps without redrawing.

In [ ]:
# RECIPE A - works with the DEFAULT inline backend, nothing to install
from IPython.display import display, clear_output

fig, ax = plt.subplots(figsize=(11, 3))
try:
    while run.alive:
        t, y = daq.live_preview()
        ax.clear()                       # (1) or the lines pile up
        if len(y):
            ax.plot(t, y, lw=0.6)
            ax.set_xlabel("board time [s]"); ax.set_ylabel("V")
            ax.set_title(f"{run.status()['seconds']:,.0f} s recorded")
        clear_output(wait=True)          # (2) wait=True = no flicker
        display(fig)
        time.sleep(0.5)
except KeyboardInterrupt:
    pass

In [ ]:
# RECIPE B - nicer: real pan/zoom, no flicker.  pip install ipympl
# Put `%matplotlib widget` in its own cell ABOVE and re-run from there.
fig, ax = plt.subplots(figsize=(11, 3))
line, = ax.plot([], [], lw=0.6)
ax.set_xlabel("board time [s]"); ax.set_ylabel("V")

try:
    while run.alive:
        t, y = daq.live_preview()
        if len(y):
            line.set_data(t, y)          # update the SAME line - no pile-up
            ax.relim(); ax.autoscale_view()
        plt.pause(0.5)                   # redraws AND sleeps
except KeyboardInterrupt:
    pass

## 7. Watch it — navigate the actual data

`live_preview` is a health check: 60 s, one channel, from RAM. `LiveScope` reads
the **files**, so it has the whole run and you can scroll back an hour.

It handles both backends itself. If you prefer your own loop, use `step()`.

In [ ]:
scope = LiveScope(OUTDIR, channel="ai0", width=5.0)
scope.show()                  # interrupt the kernel to stop

# ...or drive it yourself:
# while run.alive:
#     scope.step()

## 8. Stop

In [ ]:
run.stop()                # closes the current file cleanly
daq.ao.stop()             # waveform off, ao0 back to 0 V
print("alive:", run.alive)

## 9. Read it back

Nothing on disk has been filtered — the `.f32` is what DAQmx returned and
`edges.i64` holds every value the counter produced. Interpretation happens here,
which is why a mistake in this cell is one you can correct tomorrow.

In [ ]:
manifest, edges = load_long_run(OUTDIR)
rate = manifest[0]["nominal_rate"]
gaps = [e for e in manifest if e.get("event") == "gap"]

print("channels:", channels_of(OUTDIR))
print("segments:", sorted({e["index"] for e in manifest if "file" in e}))
print("gaps    :", len(gaps), " <- must be 0")
for g in gaps:
    print(f"   at scan {g['sample']:,}: {g['error']}")

report = edge_report(edges, nominal_rate=rate)

In [ ]:
# mmap=True keeps a multi-gigabyte segment on disk instead of in RAM
idx, y = load_segment(OUTDIR, 0, "ai0", mmap=True)
t_atomic = times_from_edges(idx, edges, scans_per_second=rate)   # cleans by default
t_board  = idx / rate

print(f"{len(y):,} scans")
print(f"atomic {t_atomic[0]:+.4f} .. {t_atomic[-1]:,.4f} s")
print(f"board  {t_board[0]:+.4f} .. {t_board[-1]:,.4f} s")

## 10. The drift — what the run was really for

The residual is the board's clock walking away from the rubidium. The staircase
is real: each edge is quantised to one scan (40 us at 25 kS/s) and the
correction is linear between edges.

In [ ]:
kept, dropped = clean_edges(edges, rate)
step = max(1, len(idx) // 200_000)          # thin for plotting only

fig, ax = plt.subplots(2, 1, figsize=(13, 7), sharex=True)
ax[0].plot(t_atomic[::step], (t_atomic - t_board)[::step] * 1e6,
           color="tab:red", lw=2)
ax[0].set_ylabel(r"$t_{atomic} - t_{board}$  [$\mu$s]")
ax[0].set_title(f"clock error {report['ppm']:+.2f} ppm"
                f"  ->  {report['seconds_per_day']:.2f} s/day")

seconds = np.arange(len(kept) - 1)
ax[1].plot(seconds, np.diff(kept), lw=1)
ax[1].axhline(rate, color="k", ls="--", lw=1, label=f"nominal {rate:,.0f}")
ax[1].set_xlabel("atomic second"); ax[1].set_ylabel("scans / atomic second")
ax[1].legend()
plt.tight_layout(); plt.show()

## 11. If something went wrong

| symptom | look at |
|---|---|
| `gaps` is not 0 | the manifest error string; usually `-200279` (PC too slow) or `-200361` (USB) |
| `edge_report` shows few edges | `daq.wiring()` — is the 1 pps on the terminal you think? |
| ppm is enormous | you are on a stale kernel; restart and re-read |
| `t_atomic` starts at 1.0 | same — `clean_edges` is not running |
| AO underflow `-200621` | `daq.ao.onboard()` was False; lower `ao.rate` or raise `ao.freq` |

Keep the raw directory. Every number above can be recomputed from it.

In [ ]:
daq.close()